# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata and inspect contents
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")
print(f"Authors: {getattr(metadata, 'author', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

In [ ]:
from pprint import pprint

print("Record sets (using @id):")
record_sets = list(dataset.record_sets()) # yields CroissantRecordSet objects

record_set_ids = []
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")
    # List field @id and column @id for each record set
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
        if hasattr(field, "columns"):
            for col in getattr(field, "columns", []):
                print(f"        Column @id: {col.id}, name: {col.name}")
    record_set_ids.append(rs.id)
if not record_sets:
    print("No explicit Croissant record sets found (some datasets only define a default record set, try loading the default).")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set, field, and column `@id`s discovered above.

In [ ]:
# Extract data from each record set using @id
dataframes = {}
if record_set_ids:
    # Use the first record set as an example
    main_record_set_id = record_set_ids[0]
    print(f"Loading records from record set: {main_record_set_id}")
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Columns in record set {main_record_set_id}:")
    print(df.columns.tolist())
    display(df.head())
else:
    # If no explicit record set, try loading top-level records
    print("Attempting to load records with record_set=None (default)")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes[None] = df
    print("Columns:", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Choose a numeric field (column) for analysis
if record_set_ids:
    # Main record set
    rs_id = main_record_set_id
else:
    # Default
    rs_id = None

df = dataframes[rs_id]
numeric_candidate_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_candidate_fields:
    numeric_field = numeric_candidate_fields[0] # Select the first numeric field
    print(f"Using numeric field: {numeric_field}")
    threshold = df[numeric_field].mean() # Use mean as an example of threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field}:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Try to group by a categorical column
    group_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
    if group_candidates:
        group_field = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped average of {numeric_field} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable group field (categorical/string) found.")
else:
    print("No numeric fields found to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram and boxplot of the numeric field
if 'numeric_field' in locals():
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field].dropna())
    plt.title(f"Boxplot of {numeric_field}")

    plt.tight_layout()
    plt.show()
    # If group candidates available, boxplot by group
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field visualized.")

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using the Croissant schema and explored its structure and data using the `mlcroissant` library. We reviewed available record sets, fields, and columns by their `@id`s, extracted the data for analysis, performed exploratory data analysis including filtering and normalization, and visualized numeric field distributions.

You can extend this notebook by exploring additional fields, performing more advanced transformations, or applying machine learning to the extracted records.